In [15]:
api_key=os.environ["OPENAI_API_KEY"]

In [16]:
import pandas as pd
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI
import json 

# Manually create DataFrame
df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie", "David"],
    "Reference": ["PMDS57838", "PMDS57837", "PMDS57836", "PMDS57837"],
    "Age": [25, 30, 35, 40],
    "Outstanding_fees": [50000, 60000, 70000, 80000],
    "Degree_level": ["NQF 9", "NQF 7", "NQF 10", "NQF 9"],
    "Commnents": ["average less than 65%, already obtained nqf level 9", "NSFAS not attached, In progressed academic record", "applied before july 2024", "satisfied all requirements"],
})

# Create LLM
llm = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
    openai_api_key=api_key  # or load from .env
)

# Create Pandas agent with dangerous code execution allowed
agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose=True,
    allow_dangerous_code=True,
    return_intermediate_steps=True,
    max_iterations=200  # default is usually 3
)


# Ask a question
response = agent.invoke("show dataframe")

dict={"code":response['intermediate_steps'][0][0].tool_input,
"output":response['output']}


#print(dict)
print(dict["output"])
print(dict["code"])



> Entering new AgentExecutor chain...
Thought: The user wants to see the contents of the dataframe `df`. The `print(df.head())` output already shows the first 5 rows, but to show the entire dataframe (if it's small), I should display it directly.
Action: python_repl_ast
Action Input: df      Name  Reference  Age  Outstanding_fees Degree_level  \
0    Alice  PMDS57838   25             50000        NQF 9   
1      Bob  PMDS57837   30             60000        NQF 7   
2  Charlie  PMDS57836   35             70000       NQF 10   
3    David  PMDS57837   40             80000        NQF 9   

                                           Commnents  
0  average less than 65%, already obtained nqf le...  
1  NSFAS not attached, In progressed academic record  
2                           applied before july 2024  
3                         satisfied all requirements  Final Answer: 

Here is the full dataframe:

|    | Name    | Reference   |   Age |   Outstanding_fees | Degree_level   | Commnents

In [13]:
from langchain_core.messages import HumanMessage

In [14]:
# v1.2.10



conversation=[]

def context(conversation, user_input):
    full_prompt=f"""
    You are working with pandas dataframe.
    
    Here is the context:
    {conversation}
    
    New question:
    {user_input}
    """

    return full_prompt

while True:
    user_input= input("You: ")
    conversation.append(f"User: {user_input}")
    if user_input in ["exit","quit"]:
        break

    
    response=agent.invoke(context(conversation, user_input))
    
    #bot="Bot:\n "+ response["output"] + response['intermediate_steps'][0][0].tool_input

    
    dict={"output":response['output'], "code":response['intermediate_steps'][0][0].tool_input}


    conversation.append("\n".join(dict['output']))
    
    print(dict['output'])
    print(exec(dict['code']))

You:  explain data




> Entering new AgentExecutor chain...
Thought: The user wants an explanation of the data in the dataframe. I should describe the columns, their likely meanings, and the type of information each row represents.
Action: python_repl_ast
Action Input: df.info()<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Name              4 non-null      str  
 1   Reference         4 non-null      str  
 2   Age               4 non-null      int64
 3   Outstanding_fees  4 non-null      int64
 4   Degree_level      4 non-null      str  
 5   Commnents         4 non-null      str  
dtypes: int64(2), str(4)
memory usage: 324.0 bytes
Thought: I now know the final answer
Final Answer: The dataframe contains information about four individuals, likely students or applicants. Here is an explanation of each column:

- Name: The person's name (string).
- Reference: A unique re

You:  quit
